# 01 — Data Preparation

**Paper title:** Maternal gut microbiome diversity and functional potential in early pregnancy are associated with large-for-gestational-age birth (SweMaMi cohort)

**Purpose:** Constructs the matched analytic dataset: 1:2 propensity score matching (MatchIt, nearest neighbour) on gestational week at sampling, maternal age and pre-pregnancy BMI and defines confounders (maternal age, pre-pregnancy BMI, parity, dietary category) and hypothesized mediators (GDM, EGWG).

**Produces:** Figure 1 (sample selection flowchart), Table 1 (baseline characteristics)


## Setup


In [ ]:
library(dplyr)
library(tidyverse)
library(readxl)
library(data.table)
library(openxlsx)
library(vegan)
library("MatchIt")
library(vegan)

In [ ]:
# Define base path 
base_path <- "data"

## 1. Load birth data, metadata, growth chart data

In [ ]:

wt_cat <- read.table(
  file.path(base_path, "weight_gest_age.csv"),
  sep = ";",
  header = TRUE
)

all_data <- read.csv(
  file.path(base_path, "10SEPT_2024_swemami.csv"),
  header = TRUE
)
# all_taxa_tab was already filtered:
# Species were retained if they reached a minimum relative abundance of 10^-5
# in at least 5% of all SweMaMi fecal samples.
all_taxa_tab <- read.table(file.path(base_path, "all_taxa_tab.tsv"),sep = "\t",header = TRUE,
  check.names = FALSE,
  quote = '"'
)

seq_yes <- read.csv(
  file.path(base_path, "all_seqd_samples.csv")
)

## 2. Apply eligibility / exclusion criteria


In [ ]:
sum(is.na(all_data$kit1.faecal_sample.barcode))
sum(is.na(all_data$kit1.faecal_sample.datetime_taken))
both_date_id_miss <- all_data %>% filter((is.na(kit1.faecal_sample.datetime_taken) & is.na(kit1.faecal_sample.barcode)))
dim(both_date_id_miss)
Any_one_miss <- all_data %>% filter((is.na(kit1.faecal_sample.datetime_taken) | is.na(kit1.faecal_sample.barcode)))
dim(Any_one_miss)
non_seq <- all_data  %>% filter(!(kit1.faecal_sample.barcode %in% seq_yes$sample_id))
dim(non_seq)


#No baseline samples n = 1466
#Missing sample ID at TP1 n = 60
#Not sequenced at TP1 n = 1507


In [ ]:
all_data_1 <- all_data %>% filter(!(is.na(kit1.faecal_sample.datetime_taken) | is.na(kit1.faecal_sample.barcode)))
dim(all_data_1)

In [ ]:
#No taxonomic data for TP1: QC failure for these samples, n = 127
setdiff(seq_yes$sample_id, colnames(all_taxa_tab))

In [ ]:
#Unknown singleton/twin status, n = 591
sum(is.na(all_data_1$Q3_X8_One_or_more_child))

In [ ]:
#Twin delivery, n = 38
table(all_data_1$Q3_X8_One_or_more_child)

In [ ]:
#TP1 samples after gestational week 20, n = 22
sum((all_data_1$pregnancy_week_of_sampling_variable < 6 | all_data_1$pregnancy_week_of_sampling_variable > 20),
  na.rm = TRUE)

In [ ]:
#Key birth details missing weight, sex and week, n = 90
sum(is.na(only_seq$GL_v)) # gestational week is missing , n = 84
sum(is.na(only_seq$Q3_X10_Boy_or_girl)) # sex is missing , n= 1
sum(is.na(all_flt_2$Birthweight_checked)) # Birth weight information is missing, n = 5

In [ ]:
#TP1 samples after gestational week 20, n = 22
sum((all_data_1$pregnancy_week_of_sampling_variable < 6 | all_data_1$pregnancy_week_of_sampling_variable > 20),
  na.rm = TRUE)

In [ ]:
# SGA group is removed after the Birth weight classification


In [ ]:
qc_fail <- setdiff(seq_yes$sample_id, colnames(all_taxa_tab))

In [ ]:
all_data_2 <- all_data_1 %>%
  filter(kit1.faecal_sample.barcode %in% seq_yes$sample_id)

all_data_3 <- all_data_2 %>%
  filter(!(kit1.faecal_sample.barcode %in% qc_fail))

all_data_4 <- all_data_3 %>%
  filter(!is.na(Q3_X8_One_or_more_child))

all_data_5 <- all_data_4 %>%
  filter(Q3_X8_One_or_more_child == "Flicka" |Q3_X8_One_or_more_child == "Pojke") 

all_data_6 <- all_data_5 %>%
  filter(!is.na(GL_v) &                       
         !is.na(Q3_X10_Boy_or_girl) &         
         !is.na(Birthweight_checked))  

all_data_final <- all_data_6 %>%
  filter(pregnancy_week_of_sampling_variable >= 6 &
         pregnancy_week_of_sampling_variable <= 20)

dim(all_data_final)

## 3. Classify neonates as LGA / AGA using the Fenton growth chart



In [ ]:
#For each live birth, select the cases that are in the 10th or 90th percentiles of their GA and sex. 
find_weight_class<-function(sex, GW, GD, weight, ref){
    #print(c(sex, GW, GD, weight))
    if(as.numeric(GD)>0){
        GW<-as.numeric(GW)+1
    }
    if(sex == "Flicka - flickor"){
        row<-which(ref$Sex=="female" & ref$GA==GW)
        if(weight <= ref$X10.ile[row]){
            return("SGA")
        } 
         else if(weight >= ref$X90.ile[row]){
             return("LGA")
         }
          else{
              return("AGA")
          }
    }
    if(sex == "Pojke - pojkar"){
        row<-which(ref$Sex=="male" & ref$GA==GW)
        if(weight <= ref$X10.ile[row]){
            return("SGA")
        } 
         else if(weight >= ref$X90.ile[row]){
             return("LGA")
         }
          else{
              return("AGA")
          }
    }
} 

In [ ]:
weight_classes<-vector(length=nrow(all_data_final))
for(i in 1:length(weight_classes)){
    weight_classes[i]<-find_weight_class(all_data_final$Q3_X10_Boy_or_girl[i],
                                         all_data_final$GL_v[i], all_data_final$GL_d[i],
                                        all_data_final$Birthweight_checked[i], 
                                         wt_cat)
}

In [ ]:
table(weight_classes)

In [ ]:
lga_babies <- all_data_final$Studienummer[which(weight_classes == "LGA")]
length(lga_babies)
aga_babies <- all_data_final$Studienummer[which(weight_classes == "AGA")]
length(aga_babies)
sga_babies <- all_data_final$Studienummer[which(weight_classes == "SGA")]
length(sga_babies)

In [ ]:
all_data_final$Birthweight_category <- weight_classes

In [ ]:
# exclude mothers who delivered SGA babies, n = 270
all_data_final <- all_data_final %>%
  filter(Birthweight_category != "SGA")

In [ ]:
# Apply eligibility/exclusion criteria after the groups classification
#Succeeding pregnancies, n = 21

double_preg_all <- all_data_final %>%
  filter(
    (Studienummer %in% lga_babies & double_participation == "Yes") |
    (Studienummer %in% aga_babies & double_participation == "Yes")
  ) %>% select(Studienummer, Q1_Personnummer, kit1.faecal_sample.barcode,Q1_Datum)
dim(double_preg_all)
head(double_preg_all)


In [ ]:
# Remove participants who participated twice
all_data_final_unique <- all_data_final %>%
  filter(double_participation != "Yes" | is.na(double_participation))

dim(all_data_final_unique)

In [ ]:
#Low richness
#Add other_taxa row to make all the columns same constant value
new_row <- c()
tot <- colSums(all_taxa_tab)
for(i in tot){
    if(i < 100) {
        new_row <- c(new_row,100 - i)
    } else if( i >=100 ){
        new_row <- c(new_row,0)
    }
}
new_row <- setNames(new_row, names(all_taxa_tab))
new_row2 <- as.data.frame(t(new_row))
rownames(new_row2) <- "Other_taxa"
best_filt_new <- rbind(all_taxa_tab,new_row2) 
filt_new_arrange <- as.data.frame(best_filt_new)
head(filt_new_arrange)
dim(filt_new_arrange)

In [ ]:
#Filter low-richness species
taxa_richness <- apply(filt_new_arrange, 2, specnumber)
less_richness <- which(taxa_richness < 120)
id_less_richness <- names(taxa_richness)[less_richness]
id_less_richness

In [ ]:
remove_id_richness <- all_data_final_unique %>% filter(kit1.faecal_sample.barcode %in% id_less_richness) %>% pull(Studienummer)
remove_id_richness

In [ ]:
all_data_final_unique_1 <- all_data_final_unique %>%
  filter(!(kit1.faecal_sample.barcode %in% remove_id_richness))

In [ ]:
#Age < 18, n = 1
all_data_final_unique_2 <- all_data_final_unique_1 %>%
  filter(age_checked >= 18)

In [ ]:
all_data_final_unique_2 <- all_data_final_unique_2 %>%
  mutate(Group = case_when(
    Birthweight_category == "LGA" ~ "Case",
    Birthweight_category == "AGA" ~ "Control"
  ))

In [ ]:
all_data_final_unique_2$Group <- factor(all_data_final_unique_2$Group,
                                        levels = c("Control", "Case"))

## 4. 1:2 propensity score matching (MatchIt)


In [ ]:
# Perform matching
set.seed(42) 
m.out <- matchit(Group ~ BMI_group_prior + age_checked + update_pregnancy_week_of_sampling, 
                 data = all_data_final_unique_2, 
                 method = "nearest",ratio = 2,caliper = c(age_checked = 4,update_pregnancy_week_of_sampling = 1 ))

In [ ]:
matched_data <- match.data(m.out)
head(matched_data )

In [ ]:
cases <- matched_data %>% filter(Group == "Case")
controls <- matched_data %>% filter(Group == "Control")
dim(cases)
dim(controls)

In [ ]:
close_ct_df <- data.frame(Case = character(),
                                  Control1 = character(),
                                  Control2 = character(),
                                  stringsAsFactors = FALSE)
head(close_ct_df)

In [ ]:
used_controls <- c()
for (i in 1:nrow(cases)) {
  case_row <- cases[i, ]
  free_controls <- controls %>% 
    filter(!(Studienummer %in% used_controls))
  
  if (nrow(free_controls) < 2) {
    message("Not enough available controls for case: ", case_row$Studienummer)
    break
  }
    free_controls$case_distance <- abs(free_controls$distance - case_row$distance)
  
  closest_controls <- free_controls %>%
    arrange(case_distance) %>%
    slice(1:2)

  close_ct_df  <- rbind(close_ct_df , data.frame(
    Case = case_row$Studienummer,
    Control1 = closest_controls$Studienummer[1],
    Control2 = closest_controls$Studienummer[2],
    stringsAsFactors = FALSE
  ))
  
  used_controls <- c(used_controls, closest_controls$Studienummer)
}
head(close_ct_df )
dim(close_ct_df )

In [ ]:
case_age <- matched_data[matched_data$Studienummer %in% close_ct_df$Case,]
new_case_age <- case_age %>% select(Studienummer,age_checked, update_pregnancy_week_of_sampling,BMI_group_prior)
# Rename multiple columns for old to new
setnames(new_case_age, old = c("Studienummer","age_checked","update_pregnancy_week_of_sampling","BMI_group_prior" ), 
         new = c("Case","Case_age","Case_week","Case_BMI"))
head(new_case_age)
dim(new_case_age)

In [ ]:
control1_age <- matched_data[matched_data$Studienummer %in% close_ct_df$Control1,]
control1_age <- control1_age %>% select(Studienummer,age_checked, update_pregnancy_week_of_sampling,BMI_group_prior)
# Rename multiple columns for old to new
setnames(control1_age, old = c("Studienummer","age_checked","update_pregnancy_week_of_sampling","BMI_group_prior"), 
         new = c("Control1","Control1_age","Control1_week","Control1_BMI"))
head(control1_age )
dim(control1_age)

In [ ]:
control2_age <- matched_data[matched_data$Studienummer %in% close_ct_df$Control2,]
control2_age <- control2_age %>% select(Studienummer,age_checked, update_pregnancy_week_of_sampling,BMI_group_prior)
# Rename multiple columns for old to new
setnames(control2_age, old = c("Studienummer","age_checked","update_pregnancy_week_of_sampling","BMI_group_prior"), 
         new = c("Control2","Control2_age","Control2_week","Control2_BMI"))
head(control2_age )
dim(control2_age)

In [ ]:
merge_1 <- merge(close_ct_df,new_case_age, by = "Case", all.x = TRUE) 
head(merge_1)
dim(merge_1)

merge_2 <- merge(merge_1,control1_age, by = "Control1", all.x = TRUE) 
head(merge_2)
dim(merge_2)

In [ ]:
merge_3 <- merge(merge_2,control2_age, by = "Control2", all.x = TRUE) 
head(merge_3)
dim(merge_3)

In [ ]:
#For some rows it was observed that age doesn't match and the age difference is high. So, I need to drop them
#No match
merge_3 <- merge(merge_2,control2_age, by = "Control2", all.x = TRUE) 
head(merge_3)
dim(merge_3 )

In [ ]:
final_df <- merge_3[abs(merge_3$Case_age - merge_3$Control1_age) <= 4 & abs(merge_3$Case_age - merge_3$Control2_age) <= 4, ]
dim(final_df)
head(final_df)

In [ ]:
final_df_1 <- final_df[abs(final_df$Case_week - final_df$Control1_week) <= 1 & abs(final_df$Case_week - final_df$Control2_week) <= 1, ]
dim(final_df_1)
head(final_df_1)

In [ ]:
final_df_2 <- final_df_1[final_df_1$Case_BMI == final_df_1$Control1_BMI & final_df_1$Case_BMI == final_df_1$Control2_BMI, ]
dim(final_df_2)
head(final_df_2)

In [ ]:
final_df_2 <- final_df %>% select(Case, Control1, Control2)
head(final_df_2)
dim(final_df_2)

In [ ]:
final_df_2 <- final_df %>% select(Case, Control1, Control2)
head(final_df_2)
dim(final_df_2)

In [ ]:
tp1_taxa <- all_taxa_tab[, colnames(all_taxa_tab) %in% all_data_final_unique_2$kit1.faecal_sample.barcode]
write.table(tp1_taxa,file.path(base_path,"tp1_taxa.tsv"),sep = "\t", row.names = FALSE) 

In [ ]:
write.table(final_df_2,file.path(base_path, "LGA_bmi_age_week_perfect_match.tsv"),sep = "\t",row.names = FALSE)

write.csv(all_data_final_unique_2,file.path(base_path, "preliminary_meta_data_tp1.csv"),row.names = FALSE)

## 5. TP2 Case-Control


In [ ]:
meta_data_tp2 <- all_data_final_unique_2 %>%
  filter(!is.na(kit2.faecal_sample.barcode))

# Keep only those whose TP2 barcode is present in the taxa table columns
meta_data_tp2 <- meta_data_tp2 %>%
  filter(kit2.faecal_sample.barcode %in% colnames(all_taxa_tab))

dim(meta_data_tp2)
tp2_barcodes <- meta_data_tp2$kit2.faecal_sample.barcode

#Not sequenced at TP2, n = 38
#No taxonomic data for TP2 QC failure for these samples n = 112

In [ ]:
tp2_barcodes <- meta_data_tp2$kit2.faecal_sample.barcode
tp2_taxa <- all_taxa_tab[, colnames(all_taxa_tab) %in% tp2_barcodes]

In [ ]:
write.csv(meta_data_tp2,file.path(base_path, "preliminary_meta_data_tp2.csv"),row.names = FALSE)
write.table(tp2_taxa,file.path(base_path, "tp2_taxa.tsv"),sep = "\t",row.names = TRUE)

## 6. Define confounders and mediators


### Generate TP1 and TP2 meta data for this study

In [ ]:
tp1_meta_data_pre <- read.csv(file.path(base_path, "preliminary_meta_data_tp1.csv"))

In [ ]:

tp1_meta_data_1 <- tp1_meta_data_pre %>% select(c('kit1.faecal_sample.barcode','Studienummer','Q1_Personnummer','kit2.faecal_sample.barcode','age_checked','Age_groups',
'BMI_prior','Primipara','excess_weight','Gest_Diabetes','Q1_healthydiet','Q2_healthydiet', 'Swedish_born', 'smoking', 'ses_score',
            'Q1_X46_Antibiotics_during_this_pregnancy', 'Q2_X6.Antibiotics_during_pregnancy', 'high_stress_Q1',  'stress_sum_score_Q2',
                   'Depression_score_sum_Q1', 'Depression_score_sum_Q2', 'GL_v','Group'))
head(tp1_meta_data_1 )

In [ ]:
# ensure factors exist and set reference explicitly
tp1_meta_data_1$Primipara <- factor(tp1_meta_data_1$Primipara, levels = c("0","1"))
tp1_meta_data_1$excess_weight <- factor(tp1_meta_data_1$excess_weight, levels = c("No","Yes"))
tp1_meta_data_1$Gest_Diabetes <- factor(tp1_meta_data_1$Gest_Diabetes,levels = c("0","1"))
tp1_meta_data_1$Q1_healthydiet <- factor(tp1_meta_data_1$Q1_healthydiet, levels = c("1","0"))
tp1_meta_data_1$Q2_healthydiet <- factor(tp1_meta_data_1$Q2_healthydiet, levels = c("1","0"))

In [ ]:
head(tp1_meta_data_1)

In [ ]:
write.csv(tp1_meta_data_1,file.path(base_path, "meta_data_tp1.csv"),row.names = FALSE)

In [ ]:
tp2_meta_data_pre <- read.csv(file.path(base_path,"preliminary_meta_data_tp2.csv", header = TRUE)
head(tp2_meta_data_pre)

In [ ]:
tp2_meta_data_1 <- tp1_meta_data_1 %<% 
                    filter(kit2.faecal_sample.barcode %in% tp2_meta_data_pre$kit2.faecal_sample.barcode)

In [ ]:
write.csv(tp2_meta_data_1,file.path(base_path, "meta_data_tp2.csv"), row.names = FALSE)

## 7. Baseline characteristics

In [ ]:
select_df_var <- read.csv("meta_data_tp1.csv",header = TRUE)

In [ ]:
select_df_varkit1.faecal_sample.barcode <- as.character(
  select_df_varkit1.faecal_sample.barcode
)

select_df_var$kit2.faecal_sample.barcode <- as.character(
select_df_var$kit2.faecal_sample.barcode
)

In [ ]:
# Check for missing values in each covariate
cat("Missing age:", sum(is.na(select_df_var$age_checked)), "\n")
cat("Missing BMI:", sum(is.na(select_df_var$BMI_prior)), "\n")
cat("Missing parity:", sum(is.na(select_df_var$Primipara)), "\n")
cat("Missing diet:", sum(is.na(select_df_var$Q1_healthydiet)), "\n")
cat("Missing excess_weight	:", sum(is.na(select_df_var$excess_weight	)), "\n")
cat("Gest_Diabetes:", sum(is.na(select_df_var$Gest_Diabetes)), "\n")

In [ ]:
select_df_var$Swedish_born <- factor(select_df_var$Swedish_born, levels = c("0","1"))
select_df_var %>%
  group_by(Group) %>%
  summarise(
    n_swedish_born = sum(Swedish_born == "1", na.rm = TRUE)
  )

In [ ]:
select_df_var %>%
  group_by(Group) %>%
  summarise(n_missing = sum(is.na(Q1_X46_Antibiotics_during_this_pregnancy)))

In [ ]:
select_df_var$Q2_X6.Antibiotics_during_pregnancy<- factor(select_df_var$Q2_X6.Antibiotics_during_pregnancy, levels = c("0","1"))
sum(is.na(select_df_var$Q2_X6.Antibiotics_during_pregnancy))

In [ ]:
select_df_var %>%
  group_by(Group) %>%
  summarise(n_missing = sum(is.na(Q2_X6.Antibiotics_during_pregnancy)))

In [ ]:
sum(is.na(select_df_var$stress_sum_score_Q2))
table(select_df_var$stress_sum_score_Q2)

In [ ]:
select_df_var %>%
  group_by(Group) %>%
  summarise(
    n_smoking = sum(smoking == "1", na.rm = TRUE)
  )

In [ ]:
select_df_var$ses_score <- factor(
  select_df_var$ses_score,
  levels = c(1, 2, 3),
  ordered = TRUE
)

In [ ]:
select_df_var$high_stress_Q1 <- factor(select_df_var$high_stress_Q1, levels = c("0","1"))
sum(is.na(select_df_var$high_stress_Q1))

In [ ]:
select_df_var$Q2_stress_binary <- factor(
  ifelse(select_df_var$stress_sum_score_Q2 >= 7, 1, 0),
  levels = c(0, 1)
)

In [ ]:
sum(is.na(select_df_var$GL_v))
select_df_var$pre_term <- factor(
  ifelse(select_df_var$GL_v < 37, 1, 0),
  levels = c(0, 1)
)

In [ ]:
select_df_var %>%
  group_by(Group) %>%
  summarise(
    mean_GA = round(mean(GL_v, 
                         na.rm = TRUE), 1),
    sd_GA = round(sd(GL_v, 
                     na.rm = TRUE), 1),
    min_GA = min(GL_v, 
                 na.rm = TRUE),
    max_GA = max(GL_v, 
                 na.rm = TRUE)
  )

In [ ]:
select_df_var$depressed_Q1 <- factor(
  ifelse(
    is.na(select_df_var$Depression_score_sum_Q1),
    NA,
    ifelse(select_df_var$Depression_score_sum_Q1 <= 11, 0, 1)
  ),
  levels = c(0, 1)
)

In [ ]:
select_df_var$depressed_Q2 <- factor(
  ifelse(
    is.na(select_df_var$Depression_score_sum_Q2),
    NA,
    ifelse(select_df_var$Depression_score_sum_Q2 <= 11, 0, 1)
  ),
  levels = c(0, 1)
)

In [ ]:
sum(is.na(select_df_var$depressed_Q2))
sum(is.na(select_df_var$depressed_Q1))

In [ ]:
select_df_var %>%
  group_by(Group) %>%
  summarise(
    n = n(),
    mean_age = round(mean(age_checked, 
                          na.rm = TRUE), 1),
    sd_age = round(sd(age_checked, 
                      na.rm = TRUE), 1),
    missing = sum(is.na(age_checked)),
    missing_pct = round(mean(is.na(age_checked)) 
                        * 100, 1)
  )

# p-value — Welch t-test
t.test(age_checked ~ Group, 
       data = select_df_var)$p.value

In [ ]:
# BMI summary statistics by group
select_df_var %>%
  group_by(Group) %>%
  summarise(
    n = n(),
    mean_BMI = round(mean(BMI_prior, 
                          na.rm = TRUE), 1),
    sd_BMI = round(sd(BMI_prior, 
                      na.rm = TRUE), 1),
    missing = sum(is.na(BMI_prior)),
    missing_pct = round(mean(is.na(BMI_prior)) 
                        * 100, 1)
  )

# p-value
t.test(BMI_prior ~ Group,
       data = select_df_var)$p.value

In [ ]:
sum(is.na(select_df_var$BMI_prior))

In [ ]:
# Create BMI category
select_df_var <- select_df_var %>%
  mutate(BMI_category = case_when(
    BMI_prior < 18.5 ~ "Underweight",
    BMI_prior >= 18.5 & BMI_prior < 25 ~ "Normal",
    BMI_prior >= 25 & BMI_prior < 30 ~ "Overweight",
    BMI_prior >= 30 ~ "Obese",
    is.na(BMI_prior) ~ NA_character_
  ))

# Set factor levels in clinical order
select_df_var$BMI_category <- factor(
  select_df_var$BMI_category,
  levels = c("Underweight", "Normal", 
             "Overweight", "Obese")
)

In [ ]:
# Count and percentage by group
select_df_var %>%
  group_by(Group) %>%
  count(BMI_category) %>%
  group_by(Group) %>%
  mutate(
    pct = round(n/sum(n) * 100, 1),
    result = paste0(n, " (", pct, "%)")
  ) %>%
  select(Group, BMI_category, result)


In [ ]:
# Missing by group
select_df_var %>%
  group_by(Group) %>%
  summarise(
    missing_n = sum(is.na(BMI_category)),
    missing_pct = round(
      mean(is.na(BMI_category)) * 100, 1)
  )

In [ ]:
tbl_bmi <- table(select_df_var$Group,
             select_df_var$BMI_category,
             useNA = "no")
print(tbl_bmi)

In [ ]:
fisher.test(tbl_bmi)

In [ ]:
pct_bmi <- prop.table(tbl_bmi, margin = 1) * 100

round(pct_bmi, 1)

In [ ]:
#preterm
tbl_preterm <- table(select_df_var$Group,
                     select_df_var$pre_term)

tbl_preterm

prop.table(tbl_preterm, margin = 1) * 100
chisq.test(tbl_preterm)

In [ ]:
#swedish_born 
tbl_swedish_born <- table(select_df_var$Group,
                     select_df_var$Swedish_born)

tbl_swedish_born
prop.table(tbl_swedish_born, margin = 1) * 100
chisq.test(tbl_swedish_born)

In [ ]:
#Primipara
tbl_parity <- table(select_df_var$Group,
                     select_df_var$Primipara)

tbl_parity
prop.table(tbl_parity, margin = 1) * 100
chisq.test(tbl_parity)

In [ ]:
#excess_weight
tbl_weight <- table(select_df_var$Group,
                     select_df_var$excess_weight)

tbl_weight
prop.table(tbl_weight, margin = 1) * 100
chisq.test(tbl_weight)

In [ ]:
select_df_var %>%
  group_by(Group) %>%
  summarise(
    missing_n = sum(is.na(excess_weight)),
    missing_pct = mean(is.na(excess_weight)) * 100
  )

In [ ]:
#Gest_Diabetes
tbl_Gest_Diabetes <- table(select_df_var$Group,
                     select_df_var$Gest_Diabetes)

tbl_Gest_Diabetes
prop.table(tbl_Gest_Diabetes, margin = 1) * 100
chisq.test(tbl_Gest_Diabetes)

In [ ]:
#ses_score
tbl_ses_score<- table(select_df_var$Group,
                     select_df_var$ses_score)

tbl_ses_score
prop.table(tbl_ses_score, margin = 1) * 100
chisq.test(tbl_ses_score)

In [ ]:
#smoking
tbl_smoking<- table(select_df_var$Group,
                     select_df_var$smoking)

tbl_smoking
prop.table(tbl_smoking, margin = 1) * 100
fisher.test(tbl_smoking)

In [ ]:
#Q1_healthydiet
tbl_Q1_healthydiet<- table(select_df_var$Group,
                     select_df_var$Q1_healthydiet)

tbl_Q1_healthydiet
prop.table(tbl_Q1_healthydiet, margin = 1) * 100
chisq.test(tbl_Q1_healthydiet)

In [ ]:
#Q1_X46_Antibiotics_during_this_pregnancy
tbl_Q1_X46_Antibiotics_during_this_pregnancy<- table(select_df_var$Group,
                     select_df_var$Q1_X46_Antibiotics_during_this_pregnancy)

tbl_Q1_X46_Antibiotics_during_this_pregnancy
prop.table(tbl_Q1_X46_Antibiotics_during_this_pregnancy, margin = 1) * 100
chisq.test(tbl_Q1_X46_Antibiotics_during_this_pregnancy)

In [ ]:
#high_stress_Q1
tbl_high_stress_Q1<- table(select_df_var$Group,
                     select_df_var$high_stress_Q1)

tbl_high_stress_Q1
prop.table(tbl_high_stress_Q1, margin = 1) * 100
chisq.test(tbl_high_stress_Q1)

In [ ]:
#Depression_score_sum_Q1
tbl_Depression_score_sum_Q1<- table(select_df_var$Group,
                     select_df_var$Depression_score_sum_Q1)

tbl_Depression_score_sum_Q1
chisq.test(tbl_Depression_score_sum_Q1)

In [ ]:
select_df_var %>%
  group_by(Group) %>%
  summarise(
    missing_n = sum(is.na(Depression_score_sum_Q1)),
    missing_pct = mean(is.na(Depression_score_sum_Q1)) * 100
  )

In [ ]:
#Q2_healthydiet
tbl_Q2_healthydiet<- table(select_df_var_2$Group,
                     select_df_var_2$Q2_healthydiet)

tbl_Q2_healthydiet
prop.table(tbl_Q2_healthydiet, margin = 1) * 100
chisq.test(tbl_Q2_healthydiet)

In [ ]:
#Q2_X6.Antibiotics_during_pregnancy
tbl_Q2_X6.Antibiotics_during_pregnancy<- table(select_df_var_2$Group,
                     select_df_var_2$Q2_X6.Antibiotics_during_pregnancy)

tbl_Q2_X6.Antibiotics_during_pregnancy
sum(is.na(select_df_var_2$Q2_X6.Antibiotics_during_pregnancy))
prop.table(tbl_Q2_X6.Antibiotics_during_pregnancy, margin = 1) * 100
chisq.test(tbl_Q2_X6.Antibiotics_during_pregnancy)

In [ ]:
#Q2_stress
tbl_Q2_stress_binary<- table(select_df_var_2$Group,
                     select_df_var_2$Q2_stress_binary)

tbl_Q2_stress_binary
prop.table(tbl_Q2_stress_binary, margin = 1) * 100
chisq.test(tbl_Q2_stress_binary)

In [ ]:
#depressed_Q2
tbl_depressed_Q2<- table(select_df_var_2$Group,
                     select_df_var_2$depressed_Q2)

tbl_depressed_Q2
prop.table(tbl_depressed_Q2, margin = 1) * 100
chisq.test(tbl_depressed_Q2)